# Now we make our code configuration base

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.functions import (
    col, lower, when, to_timestamp, row_number, desc
)

# ================= CONFIG =================

CATALOG = "elixir"
BRONZE_SCHEMA = "brz"
SILVER_SCHEMA = "slv"
PIPELINE_NAME = "topdesk_silver_incidents"
META_SCHEMA = "meta"
STATE_TABLE = f"{CATALOG}.{META_SCHEMA}.pipeline_state"

BRONZE_TABLE = f"{CATALOG}.{BRONZE_SCHEMA}.topdesk_incidents_raw"
SILVER_TABLE = f"{CATALOG}.{SILVER_SCHEMA}.topdesk_incidents"

# DQ
REPORT_TABLE = f"{CATALOG}.{META_SCHEMA}.topdesk_incidents_report"
QUARANTINE_TABLE = f"{CATALOG}.{META_SCHEMA}.dq_quarantine"


MANDATORY_FIELDS = ["incident_id", "created_date", "ticket_status", "category"]

CONFIG = {
    "status_mapping": {
        "Closed": "Closed",
        "Completed": "Closed",
        "Cancelled": "Closed",
        "New": "Open",
        "Assigned": "Open",
        "In Progress": "Open",
        "Waiting for Supplier": "Open",
        "Waiting for Caller": "Open",
        "Waiting for Approval": "Open",
        "Waiting for Release": "Open",
        "Reopened": "Open",
        "Response received": "Open"
    },
    "category_mapping": [
        {
            "pattern": "oca unifield sup",
            "OC": "OCA",
            "business": "UniField",
            "domain": "Supply"
        },
        {
            "pattern": "ocg unifield sup",
            "OC": "OCG",
            "business": "UniField",
            "domain": "Supply"
        }
    ]
}


In [0]:
def get_last_processed(pipeline_name:str):
    rows = spark.sql(f"""
            SELECT last_run
            FROM {STATE_TABLE}
            WHERE pipeline = '{pipeline_name}'
            AND endpoint = 'incidents'
            LIMIT 1
            """).collect()
    return rows[0]["last_run"] if rows else None


def update_last_processed(pipeline_name:str, ts:str):
    spark.sql(f"""
        MERGE INTO {STATE_TABLE} AS t
        USING (
            SELECT '{pipeline_name}' AS pipeline,
                '{ts}' AS last_run,
                'incidents' AS endpoint
        ) AS s
        ON t.pipeline = s.pipeline AND t.endpoint = s.endpoint
        WHEN MATCHED THEN UPDATE SET t.last_run = s.last_run
        WHEN NOT MATCHED THEN INSERT *
        """)


    

In [0]:
def parse_bronze(df, last_processed):
    # filter only new bronze records
    if last_processed:
        df = df.filter(F.col("ingestion_time") > last_processed)

    if df.count() == 0:
        print("[INFO] No new records to process")
        return None

    # infer schema from sample
    samples = (
        df.select("raw_json")
        .where(F.col("raw_json").isNotNull())
        .limit(200)
        .collect()
    )

    json_list = [r[0] for r in samples]
    schema = (
        spark.range(1)
        .select(
            F.schema_of_json(
                F.lit("[" + ",".join(json_list) + "]")
            ).alias("schema")
        )
        .collect()[0]["schema"]
    )

    parsed_df = (
        df
        .withColumn("data", F.from_json("raw_json", schema))
        .withColumn("data", F.col("data")[0])
        .select("data.*", "ingestion_time")
    )

    print(f"[INFO] Parsed {parsed_df.count()} bronze records")
    return parsed_df



In [0]:
def clean(df):
    return df.select(
        col("id").alias("incident_id"),
        col("category.name").alias("category"),
        col("subcategory.name").alias("subcategory"),
        col("caller.dynamicName").alias("caller_name"),
        col("caller.email").alias("caller_email"),
        col("caller.branch.name").alias("country_program"),
        col("processingStatus.name").alias("status"),
        col("priority.name").alias("priority"),
        col("urgency.name").alias("urgency"),
        col("callerBranch.id").alias("caller_branch_id"),
        col("callerBranch.name").alias("caller_branch_name"),
        to_timestamp("creationDate").alias("created_date"),
        to_timestamp("modificationDate").alias("modification_ts"),
        col("ingestion_time")
    )

In [0]:
def validate(df):
    total = df.count()

    # build null condition for all mandatory fields
    invalid_condition = F.lit(False)
    for field in MANDATORY_FIELDS:
        invalid_condition = invalid_condition | F.col(field).isNull()

    valid_df = df.filter(~invalid_condition)
    dropped = total - valid_df.count()

    if dropped > 0:
        print(f"[WARN] Dropped {dropped} invalid records out of {total}")
    else:
        print(f"[INFO] All {total} records passed validation")

    return valid_df

In [0]:
def apply_status_mapping(df):
    mapping = CONFIG["status_mapping"]
    expr = None
    for system_status, business_status in mapping.items():
        condition = col("status") == system_status
        expr = when(condition, business_status) if expr is None else expr.when(condition, business_status)
    return df.withColumn("ticket_status", expr)


def apply_category_mapping(df):
    rules = CONFIG["category_mapping"]
    expr_oc = expr_business = expr_domain = None
    for rule in rules:
        condition = lower(col("category")).contains(rule["pattern"])
        if expr_oc is None:
            expr_oc = when(condition, rule["OC"])
            expr_business = when(condition, rule["business"])
            expr_domain = when(condition, rule["domain"])
        else:
            expr_oc = expr_oc.when(condition, rule["OC"])
            expr_business = expr_business.when(condition, rule["business"])
            expr_domain = expr_domain.when(condition, rule["domain"])
    return (
        df
        .withColumn("OC", expr_oc)
        .withColumn("business", expr_business)
        .withColumn("domain", expr_domain)
    )


def enrich(df):
    df = apply_status_mapping(df)
    df = apply_category_mapping(df)
    return df

##Validation

In [0]:

from dq_runner import run_dq
import config.rules_topdesk_incidents as rules_topdesk_incidents



In [0]:
def deduplicate(df):
    window = Window.partitionBy("incident_id").orderBy(desc("modification_ts"))
    return (
        df
        .withColumn("rn", row_number().over(window))
        .filter("rn = 1")
        .drop("rn")
    )


def build_final_table(df):
    return df.select(
        col("incident_id").alias("ticket_id"),
        col("created_date"),
        col("OC"),
        col("business"),
        col("domain"),
        col("country_program"),
        col("priority"),
        col("urgency"),
        col("category"),
        col("subcategory"),
        col("ticket_status"),
        col("caller_email"),
        col("caller_name"),
        col("modification_ts")
    )

def filter_in_scope(df):
    return df.filter(F.col("OC").isNotNull())

In [0]:
def write_to_silver(df):
    if not spark.catalog.tableExists(SILVER_TABLE):
        print(f"[INFO] Creating silver table {SILVER_TABLE}")
        df.write.format("delta").mode("overwrite").saveAsTable(SILVER_TABLE)
    else:
        df.createOrReplaceTempView("silver_updates")
        spark.sql(f"""
            MERGE INTO {SILVER_TABLE} t
            USING silver_updates s
            ON t.ticket_id = s.ticket_id
            WHEN MATCHED THEN UPDATE SET *
            WHEN NOT MATCHED THEN INSERT *
        """)

    print(f"[INFO] Written {df.count()} records to {SILVER_TABLE}")

In [0]:
def main():
    print("[PIPELINE] Starting silver incidents")

    last_processed = get_last_processed(PIPELINE_NAME)
    print(f"[INFO] Last processed: {last_processed or 'None - full load'}")

    # 1. parse
    parsed_df = parse_bronze(spark.table(BRONZE_TABLE), last_processed)
    if parsed_df is None:
        print("[PIPELINE] Nothing to process. Done.")
        return

    # 2. clean
    clean_df = clean(parsed_df)

    # 3. enrich
    enriched_df = enrich(clean_df)

    # 4. validate
    valid_df = validate(enriched_df)

    # 5. filter in scope
    scoped_df = filter_in_scope(valid_df)

    # 6. deduplicate
    dedup_df = deduplicate(scoped_df)

    # 7. final select
    final_df = build_final_table(dedup_df)

    # 8.  DQ framework - runs on final, clean DataFrame
    dq_result = run_dq(
        spark            = spark,
        rules_config     = rules_topdesk_incidents,
        pipeline_name    = PIPELINE_NAME,
        report_table     = REPORT_TABLE,
        quarantine_table = QUARANTINE_TABLE,
        input_df         = final_df,        # ← pass directly, no re-read from table
    )
    # 9. write - only if DQ passed
    if dq_result["passed"]:
        write_to_silver(final_df)
    else:
        raise Exception(
            f"[DQ] Validation failed with {dq_result['error_failures']} error(s) "
            f"on run {dq_result['run_id']}. Records written to quarantine. "
            f"Silver table NOT updated."
        )

    # 10. update state
    max_ingestion_time = (
        parsed_df
        .agg(F.max("ingestion_time").alias("max_ts"))
        .collect()[0]["max_ts"]
    )
    update_last_processed(PIPELINE_NAME, str(max_ingestion_time))
    print(f"[INFO] Updated last_processed → {max_ingestion_time}")

    print("=" * 60)
    print("[PIPELINE] Silver incidents complete")
    print("=" * 60)

main()

In [0]:
from pyspark.sql import Row
from datetime import datetime
test_df = spark.createDataFrame([
    Row(ticket_id="T001", ticket_status="open",       created_date=datetime(2024,1,1), modification_ts=datetime(2024,1,2)),
    Row(ticket_id="T001", ticket_status="open",       created_date=datetime(2024,1,1), modification_ts=datetime(2024,1,2)),  # duplicate id
    Row(ticket_id=None,   ticket_status="open",       created_date=datetime(2024,1,1), modification_ts=datetime(2024,1,2)),  # null id
    Row(ticket_id="T003", ticket_status=None,         created_date=datetime(2024,1,1), modification_ts=datetime(2024,1,2)),  # null status
    Row(ticket_id="T004", ticket_status="open",       created_date=datetime(2024,1,1), modification_ts=datetime(2023,1,1)),  # modification before creation
    Row(ticket_id="T005", ticket_status="INVALIDSTAT",created_date=datetime(2024,1,1), modification_ts=datetime(2024,1,2)),  # bad status
])

In [0]:
dq_result = run_dq(
            spark            = spark,
            rules_config     = rules_topdesk_incidents,
            pipeline_name    = PIPELINE_NAME,
            report_table     = REPORT_TABLE,
            quarantine_table = QUARANTINE_TABLE,
            input_df         = test_df,        # ← pass directly, no re-read from table
        )

In [0]:
%sql
DROP TABLE IF EXISTS elixir.slv.topdesk_incidents_report;
DROP TABLE IF EXISTS elixir.meta.dq_quarantine;